# Harry Potter RAG — Book 1

A brief example of a simple workflow of a RAG pipeline using the famous *Harry Potter & The Philosopher's Stone* by J.K. Rowling, including:

- Text processing
- Chunking
- Embedding
- VectorDB management
- Q&A
- Evaluation

Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published.

## Import dependencies

In [1]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm

from dotenv import load_dotenv


In [2]:
def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))
os.getenv("AWS_REGION")


'ap-northeast-1'

## Setup Bedrock for LLM

Consist of 2 functions:

1. Conversation: Use it for bedrock health check, chat function, Q&A for RAG as well as evaluation using LLM-as-a-judge
2. Embedding: text embedding of Harry Potter's content for RAG

In [3]:
class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))


## Import data (the book)

From `get_toc()` we know the pdf is structured with chapter+1 as the `pymupdf` index number

In [4]:
doc = pymupdf.open(str(DATA / "harrypotter.pdf"))
toc = doc.get_toc()
toc[:25]


[[1, 'Title Page', 2],
 [1, 'Pottermore.com', 4],
 [1, 'Contents', 6],
 [1, 'Harry Potter and the Sorcerer’s Stone', 7],
 [2, 'Chapter 1 - The Boy Who Lived', 12],
 [2, 'Chapter 2 - The Vanishing Glass', 26],
 [2, 'Chapter 3 - The Letters from No One', 37],
 [2, 'Chapter 4 - The Keeper of the Keys', 50],
 [2, 'Chapter 5 - Diagon Alley', 62],
 [2, 'Chapter 6 - The Journey from Platform Nine and Three-quarters', 85],
 [2, 'Chapter 7 - The Sorting Hat', 107],
 [2, 'Chapter 8 - The Potions Master', 123],
 [2, 'Chapter 9 - The Midnight Duel', 133],
 [2, 'Chapter 10 - Halloween', 150],
 [2, 'Chapter 11 - Quidditch', 164],
 [2, 'Chapter 12 - The Mirror of Erised', 176],
 [2, 'Chapter 13 - Nicolas Flamel', 194],
 [2, 'Chapter 14 - Norbert the Norwegian Ridgeback', 205],
 [2, 'Chapter 15 - The Forbidden Forest', 217],
 [2, 'Chapter 16 - Through the Trapdoor', 234],
 [2, 'Chapter 17 - The Man with Two Faces', 257],
 [2, 'Copyright page', 275],
 [1, 'Harry Potter and the Chamber of Secrets', 276]

TOC contains copyright and other extra pages not required in RAG building.

Example of actual book content. Requires cleaning of `\n` separator as well as joining pages to avoid breaking words / context.

In [5]:
doc[12-1].get_text()[:250]

'M\n \nCHAPTER  ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbec'

## Building index / metadata

In [6]:
# Get book names
book_names = [
    chapter[1] for chapter in toc if chapter[1].startswith("Harry Potter")
]

In [7]:
_CHAPTER_RE = re.compile(r"^Chapter\s+(\d+)\s*-\s*(.+)$", re.IGNORECASE)
_CONTENT_SPECIAL = {"Prologue", "Epilogue"}
_BACK_MATTER = "Copyright page"

book_names = [t[1] for t in toc if t[1].startswith("Harry Potter")]
index = []
book_num = 0

def _add_section(book_num, chapter_num, name, page)->None:
    index.append({
        "book_num": book_num,
        "book_name": book_names[book_num - 1],
        "chapter_num": chapter_num,      # None for Prologue/Epilogue
        "chapter_name": name,
        "chapter_start_page_num": page,
        "chapter_end_page_num": None,
    })

for level, title, page in toc:
    title = title.strip()

    if title in book_names:              # book title row -> just bump the counter
        book_num += 1
        continue

    if title == _BACK_MATTER:            # back matter CAPS the previous section
        if index and index[-1]["chapter_end_page_num"] is None:
            index[-1]["chapter_end_page_num"] = page - 1
        continue

    if title in _CONTENT_SPECIAL:
        _add_section(book_num, None, title, page)
        continue

    m = _CHAPTER_RE.match(title)
    if m:
        _add_section(book_num, int(m.group(1)), title, page)

# any section not already capped by back matter ends before the next content section
for prev, nxt in zip(index, index[1:]):
    if prev["chapter_end_page_num"] is None:
        prev["chapter_end_page_num"] = nxt["chapter_start_page_num"] - 1

# final section (book 7's Epilogue) if no Copyright follows it
if index and index[-1]["chapter_end_page_num"] is None:
    index[-1]["chapter_end_page_num"] = doc.page_count

index[:5]

[{'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 1,
  'chapter_name': 'Chapter 1 - The Boy Who Lived',
  'chapter_start_page_num': 12,
  'chapter_end_page_num': 25},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 2,
  'chapter_name': 'Chapter 2 - The Vanishing Glass',
  'chapter_start_page_num': 26,
  'chapter_end_page_num': 36},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 3,
  'chapter_name': 'Chapter 3 - The Letters from No One',
  'chapter_start_page_num': 37,
  'chapter_end_page_num': 49},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 4,
  'chapter_name': 'Chapter 4 - The Keeper of the Keys',
  'chapter_start_page_num': 50,
  'chapter_end_page_num': 61},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 5,
  'chapter_name': 'Chapter 5 - Diagon Alley',
  'chapter_start_pag

## Get text by chapter

In [8]:
def get_text_from_chapter(chapter)->str:
    start = chapter["chapter_start_page_num"] - 1    # PymuPDF is 1-based on page index
    end = chapter["chapter_end_page_num"]          

    lines = []
    for page in doc[start:end]:
        lines.extend(page.get_text().split("\n"))

    text = ""
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if text and not text.endswith("-"):        # wrapped hyphen special case
            text += " "
        text += line

    return re.sub(r"\s+", " ", text).strip()       # collapse any leftover runs
        
text_data = []

for chapter in index:
    if chapter['book_num'] == 1:
        text_data.append(get_text_from_chapter(chapter))


In [9]:
len(text_data)

17

In [10]:
with open(DATA / "harrypotter-rawtext.json", "w", encoding="utf-8") as f:
    json.dump(text_data, f)


## Chunking text by chapter - Naive Chunking

`tiktoken` is a useful library to split text by token size. LangChain example:  https://docs.langchain.com/oss/python/integrations/splitters/split_by_token

`tiktoken` doc: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb

In [11]:
encoding = tiktoken.get_encoding("o200k_base")
len(encoding.encode(text_data[0]))

6327

In [12]:
# Check token/text count per chapter
list(zip(
    [i for i in range(1,18)],
    [len(encoding.encode(text)) for text in text_data],
    [len(text.split()) for text in text_data]
))

[(1, 6327, 4727),
 (2, 4475, 3505),
 (3, 5016, 3924),
 (4, 5246, 3813),
 (5, 9279, 6717),
 (6, 8518, 6387),
 (7, 6033, 4537),
 (8, 3962, 3071),
 (9, 6836, 4989),
 (10, 5651, 4322),
 (11, 4641, 3400),
 (12, 7215, 5569),
 (13, 4420, 3245),
 (14, 4795, 3535),
 (15, 7085, 5187),
 (16, 8862, 6611),
 (17, 7764, 5756)]

Function to chunk text by sliding window with overlaps

In [13]:
def chunk_text_by_sliding_window(
    text,
    chunk_token_size=512,
    overlap_ratio=0.15,
) -> list[dict]:
    # Guard bad params (a step of 0 would loop forever).
    if not 0 <= overlap_ratio < 1:
        raise ValueError("overlap_ratio must be in [0, 1)")

    encoded_text = encoding.encode(text)
    total_tokens = len(encoded_text)

    step = max(int(chunk_token_size * (1 - overlap_ratio)), 1)
    overlap_tokens = chunk_token_size - step

    output = []
    token_pointer = 0

    while token_pointer < total_tokens:
        upper_token_pointer = min(token_pointer + chunk_token_size, total_tokens)

        # Tail window that adds no new tokens (already covered by the overlap)
        if token_pointer > 0 and upper_token_pointer - token_pointer <= overlap_tokens:
            break

        decoded_text = encoding.decode(encoded_text[token_pointer:upper_token_pointer])
        output.append({
            "text": decoded_text,
            "token_start": token_pointer,
            "token_count": upper_token_pointer - token_pointer,
        })

        token_pointer += step

    return output


Chunk by chapter of book 1 only, create `jsonl` file

In [14]:
chunk_text_output = []
def generate_chunk_id(chapter_num, idx)->str:
    return f"hp1-ch{chapter_num:02d}-{idx:04d}"

for _, text_chapter in enumerate(text_data):
    chapter_num = _ + 1
    chunked_texts = chunk_text_by_sliding_window(text_chapter)
    chapter_details = [
        i for i in index 
        if i["book_num"] == 1 and i["chapter_num"] == chapter_num
    ][0]

    chunk_id_count = 1
    for segment_details in chunked_texts:
        chunk_id = generate_chunk_id(
            chapter_num=chapter_num, idx=chunk_id_count
        )
        chunk_details = chapter_details | {
            "chunk_id": chunk_id,
            "token_start": segment_details.get("token_start", None),
            "token_count": segment_details.get("token_count", None),
            "text": segment_details.get("text", None)
        }
        chunk_text_output.append(chunk_details)
            
        chunk_id_count += 1


In [16]:
with open(DATA / "harrypotter-chunks.jsonl", "w", encoding="utf-8") as f:
    for chunk in chunk_text_output:
        # JSONL: only 1 object 1 line
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")


## Embedding into VectorDB

Using AWS Bedrock's `amazon.titan-embed-text-v2:0` embedding model. Store vectors into local `chromadb`

## Setup ChromaDB

In [ ]:
client = chromadb.PersistentClient(path=str(LABS / ".chroma")))

